# Step 5 — Fine-tuning the Semantic Skill Matcher

**Thesis context.** HR Helper Module 3 — semantic skill matcher. Step 4 locked
a zero-shot baseline of macro F1 = 0.151 on the held-out eval corpus. This
notebook fine-tunes `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`
on the Step 3 distantly-supervised pairs so the Linker can lift retrieval F1
to ≥ 0.55 (DECISIONS.md D7).

**Workflow.** This is **Phase β** of the three-phase Step 5 workflow:

* **Phase α** (already done, on the dev box): produce the training bundle.
* **Phase β** (this notebook): train on Colab T4, save the checkpoint to Drive.
* **Phase γ** (next, back on the dev box): import the checkpoint, re-run the
  Step 4 baseline, decide Step 6 vs Step 5b.

**Inputs.** A single zip uploaded to `MyDrive/hr_helper/colab_bundle/` containing:

* `train.jsonl` / `val.jsonl` — Step 3 pairs (3 458 positives + 10 374 hard negatives).
* `esco_concepts.json` — full ESCO + custom-overlay set with pre-rendered
  bounded-a concept text per concept (so we do not need the `skill_matcher`
  package on Colab).
* `training_data.py` — the bundled converter that turns pairs into SBERT-ready rows.
* `bundle_manifest.json` — provenance (counts, hashes, seed, mode).

**Outputs.** A single zip on `MyDrive/hr_helper/checkpoints/<run_name>/`:

* The fine-tuned SentenceTransformer checkpoint (full SBERT layout).
* `<run_name>_manifest.json` — training provenance (epochs, batch, lr, final MRR).
* `loss_curve.png` — training-loss + val-MRR plot.

**Runtime.** ~25-45 min on a T4 with the default config (MNRL, 3 epochs,
batch 32). Colab Pro session limits are well above this.

**What Silviu does after this notebook finishes.**

1. Download `<run_name>.zip` from Drive to the dev box.
2. `python scripts/import_finetuned_model.py --zip <path> --run-name <name>`.
3. `python scripts/run_finetuned_baseline.py --out-report reports/skill_matcher_finetuned_<date>.md`.

## 1. Configuration

All over-rideable knobs in one place. Defaults match the Step 5 Pre-Flight
recommendation (MNRL, 3 epochs, batch 32, lr 2e-5). The seed and the
concept-text format are **not** over-rideable — they are locked by
`DECISIONS.md` D5/D10.

In [ ]:
from datetime import datetime, timezone

# --- Drive layout ---
DRIVE_ROOT = '/content/drive/MyDrive/hr_helper'
BUNDLE_DIR = f'{DRIVE_ROOT}/colab_bundle'           # input bundle lives here
CHECKPOINTS_DIR = f'{DRIVE_ROOT}/checkpoints'       # output checkpoints land here

# --- Run name ---
# Loss-variant + iteration + UTC date (per Pre-Flight Q3 / Silviu's go-default).
RUN_NAME = 'mnrl_sw_v1_' + datetime.now(tz=timezone.utc).strftime('%Y%m%d_%H%M')
OUTPUT_DIR = f'{CHECKPOINTS_DIR}/{RUN_NAME}'

# --- Training hyperparameters (Pre-Flight §3) ---
MODE = 'pairs_mnrl'        # 'pairs_mnrl' (default) or 'triplets' (ablation)
EPOCHS = 3                  # SBERT short-fine-tune budget
BATCH_SIZE = 32             # T4 fits this for MiniLM; halve on OOM (auto)
LEARNING_RATE = 2e-5        # SBERT default; multilingual encoders dislike higher
WEIGHT_DECAY = 0.01         # AdamW default
WARMUP_RATIO = 0.10         # 10% of total steps
MAX_SEQ_LENGTH = 128        # anchors are short; cap is defensive
EVAL_EVERY_N_STEPS = 200    # IR evaluator tick frequency

# --- Locked decisions (DO NOT change in this notebook) ---
BASE_MODEL = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
SEED = 42                                # D10
CONCEPT_TEXT_FORMAT = 'bounded-a'        # D-amendment 2026-05-16

print(f'Run name: {RUN_NAME}')
print(f'Bundle dir: {BUNDLE_DIR}')
print(f'Output dir: {OUTPUT_DIR}')
print(f'Mode: {MODE}, epochs={EPOCHS}, batch={BATCH_SIZE}, lr={LEARNING_RATE}')

## 2. Mount Google Drive and prepare the folder structure

`MyDrive/hr_helper/` is empty on first run — these cells create the layout
and verify the bundle is present. Re-runnable safely (idempotent `mkdir -p`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
os.makedirs(BUNDLE_DIR, exist_ok=True)
os.makedirs(CHECKPOINTS_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Verify the bundle has been uploaded.
required = ['train.jsonl', 'val.jsonl', 'esco_concepts.json',
            'training_data.py', 'bundle_manifest.json']
missing = [f for f in required if not os.path.exists(f'{BUNDLE_DIR}/{f}')]
if missing:
    raise FileNotFoundError(
        f'Bundle is missing files: {missing}. Upload the colab_bundle.zip ',
        f'contents to {BUNDLE_DIR} before running this notebook.'
    )
print(f'Bundle OK: {required}')

## 3. Install pinned dependencies

`sentence-transformers>=3.0,<4` matches the `[ml]` extra range in
`nlp-service/pyproject.toml` (`>=2.5,<4`). It pulls a compatible `torch`
and `transformers` transitively. Colab's pre-installed `torch` is left in
place if the version satisfies sentence-transformers.

In [ ]:
!pip install -q 'sentence-transformers>=3.0,<4'

## 4. Imports and seeded RNGs

Every random source is pinned to seed 42 (D10): Python's `random`,
NumPy, PyTorch, CUDA, `transformers.set_seed`, and CuDNN's deterministic
mode. Determinism costs a little throughput on the T4 but the thesis
defence answer hinges on bit-stable reproducibility.

In [ ]:
# Disable wandb / Hugging-Face hub reporting BEFORE any transformers /
# sentence_transformers import. SBERT 3.x uses transformers.Trainer
# under the hood which auto-detects wandb if installed and pops an
# interactive login prompt mid-training. We don't need experiment
# tracking here — the run manifest + loss-curve PNG are our provenance.
import os
os.environ['WANDB_DISABLED'] = 'true'
os.environ['WANDB_MODE'] = 'disabled'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = '1'

import json, os, random, sys, time, zipfile
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader

from sentence_transformers import (
    SentenceTransformer, InputExample, losses, util,
)
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from transformers import set_seed

# --- Seed everything ---
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
set_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Torch {torch.__version__}, device={device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: no GPU detected. Switch Runtime → Change runtime type → T4 GPU.')

## 5. Load the training bundle

Reads the four JSON/JSONL artefacts. The `esco_concepts.json` payload
already carries pre-rendered bounded-a `concept_text` per concept — we
use those strings verbatim as the positive side of each training pair.

The bundled `training_data.py` is shipped for provenance/traceability
but is **not** imported here: it depends on `skill_matcher.dataset` /
`skill_extractor` / `structlog`, none of which are on Colab. The
notebook's inline `build_input_examples` (cell below) reimplements the
same conversion logic.

In [ ]:
# Note: the bundled training_data.py is shipped for traceability
# (it's the same code that runs on the dev box for pair construction),
# but Colab cannot import it directly because it depends on
# skill_matcher.dataset / skill_extractor / structlog. The notebook's
# inline build_input_examples (cell below) reimplements the same logic.

# Bundle manifest — provenance hash chain back to Step 3.
with open(f'{BUNDLE_DIR}/bundle_manifest.json', 'r', encoding='utf-8') as fp:
    bundle_manifest = json.load(fp)
print('Bundle manifest:')
print(json.dumps(bundle_manifest, indent=2)[:1500])

# Sanity: the mode encoded in the bundle should match this notebook's setting.
if bundle_manifest['mode'] != MODE:
    print(f'WARNING: bundle.mode={bundle_manifest["mode"]!r} but notebook MODE={MODE!r}.')
    print('Using notebook MODE — the bundle carries enough data for both.')

In [ ]:
# ESCO concept set (pre-rendered concept_text per concept).
with open(f'{BUNDLE_DIR}/esco_concepts.json', 'r', encoding='utf-8') as fp:
    esco_payload = json.load(fp)

assert esco_payload['concept_text_format'] == CONCEPT_TEXT_FORMAT, (
    f'Bundle concept_text_format {esco_payload["concept_text_format"]!r} '
    f'does not match locked format {CONCEPT_TEXT_FORMAT!r}'
)

# Index by URI for fast lookup. We materialise the EscoConcept-like rows
# as a small dataclass so build_positive_text_from_uri (which calls into
# format_concept_text) has the shape it expects. BUT — since the bundle
# already carries the rendered concept_text, we can shortcut: just build
# uri -> concept_text dict and skip re-rendering.
concept_text_by_uri = {
    c['uri']: c['concept_text']
    for c in esco_payload['concepts']
}
print(f'Loaded {len(concept_text_by_uri)} concepts.')

In [ ]:
# Load the train/val pairs. The bundle ships them as JSONL produced by
# skill_matcher.dataset.save_training_dataset — header row + one pair per line.
def load_jsonl_pairs(path):
    """Yield raw pair dicts (skip the header row)."""
    pairs = []
    with open(path, 'r', encoding='utf-8') as fp:
        for line in fp:
            row = json.loads(line)
            if row.get('record_kind') == 'header':
                continue
            pairs.append(row)
    return pairs

train_rows = load_jsonl_pairs(f'{BUNDLE_DIR}/train.jsonl')
val_rows = load_jsonl_pairs(f'{BUNDLE_DIR}/val.jsonl')
print(f'Train pairs: {len(train_rows):,} (positives + hard negatives)')
print(f'Val pairs:   {len(val_rows):,} (positives + hard negatives)')

## 6. Convert pairs → SBERT InputExamples

For `mode='pairs_mnrl'` we emit one `InputExample([anchor, positive])` per
positive pair. Hard negatives are not emitted — `MultipleNegativesRankingLoss`
draws its negatives from other positives in the same mini-batch (~31 negatives
per anchor at batch 32). This is the canonical SBERT recipe for paired data.

For `mode='triplets'` we would emit one `InputExample([anchor, positive,
hard_neg])` per mined hard negative — ~10 374 rows per epoch. Reserved for
the ablation fallback.

In [ ]:
ANCHOR_HARD_CAP = 256  # mirrors training_data._ANCHOR_HARD_CAP_CHARS

def build_anchor_text(row):
    # Step 5.2: prefer query_text (sliding-window mode) when set;
    # fall back to legacy concat for Step 5 round-1 datasets.
    if row.get('query_text'):
        return row['query_text'].strip()[:ANCHOR_HARD_CAP]
    s = (row.get('context_before', '') + row['text_span'] + row.get('context_after', '')).strip()
    return s[:ANCHOR_HARD_CAP]

def build_input_examples(rows, mode):
    """Returns a list of sentence_transformers.InputExample."""
    # Index positives by pair_id for triplet mode lookup.
    positives = [r for r in rows if r['pair_type'] == 'positive']
    negatives_by_pid = {}
    for r in rows:
        if r['pair_type'] == 'hard_negative':
            negatives_by_pid.setdefault(r['paired_with_positive_id'], []).append(r)

    examples = []
    skipped_no_uri = 0
    if mode == 'pairs_mnrl':
        for p in positives:
            pos_text = concept_text_by_uri.get(p['esco_uri'])
            if pos_text is None:
                skipped_no_uri += 1
                continue
            examples.append(InputExample(texts=[build_anchor_text(p), pos_text]))
    elif mode == 'triplets':
        for p in positives:
            pos_text = concept_text_by_uri.get(p['esco_uri'])
            if pos_text is None:
                skipped_no_uri += 1
                continue
            anchor = build_anchor_text(p)
            for neg in negatives_by_pid.get(p['pair_id'], []):
                neg_text = concept_text_by_uri.get(neg['esco_uri'])
                if neg_text is None:
                    skipped_no_uri += 1
                    continue
                examples.append(InputExample(texts=[anchor, pos_text, neg_text]))
    else:
        raise ValueError(f'Unknown mode {mode!r}')
    if skipped_no_uri:
        print(f'  Skipped {skipped_no_uri} rows: URI missing from concept set.')
    return examples

train_examples = build_input_examples(train_rows, MODE)
val_examples = build_input_examples(val_rows, MODE)
print(f'Train InputExamples: {len(train_examples):,}')
print(f'Val InputExamples:   {len(val_examples):,}')

# Shuffle with seeded RNG.
shuffle_rng = random.Random(SEED)
shuffle_rng.shuffle(train_examples)
print(f'Sample anchor: {train_examples[0].texts[0]!r}')
print(f'Sample positive: {train_examples[0].texts[1][:200]!r}...')

## 7. Anchor token-length sanity check

`MAX_SEQ_LENGTH=128` only matters if real anchors fit inside it. If the
99th percentile exceeds 100 tokens, the tokenizer will truncate informative
context and we should bump the cap. The cell below tokenises a sample and
prints the distribution.

In [ ]:
from sentence_transformers.models import Transformer
_tmp_model = SentenceTransformer(BASE_MODEL)
_tmp_model.max_seq_length = MAX_SEQ_LENGTH
_tokenizer = _tmp_model.tokenizer

# Sample 1 000 anchors to keep this fast.
sample = train_examples[:1000]
anchor_lens = [len(_tokenizer.encode(ex.texts[0], add_special_tokens=True)) for ex in sample]
pos_lens = [len(_tokenizer.encode(ex.texts[1], add_special_tokens=True)) for ex in sample]

print('Anchor token lengths (n=1000):')
print(f'  mean={np.mean(anchor_lens):.1f}  p50={np.percentile(anchor_lens, 50):.0f}  p99={np.percentile(anchor_lens, 99):.0f}  max={max(anchor_lens)}')
print('Positive token lengths (n=1000):')
print(f'  mean={np.mean(pos_lens):.1f}  p50={np.percentile(pos_lens, 50):.0f}  p99={np.percentile(pos_lens, 99):.0f}  max={max(pos_lens)}')

if np.percentile(anchor_lens, 99) > 100:
    print('Warning: anchor p99 > 100 tokens; consider bumping MAX_SEQ_LENGTH to 192.')

# Free the temp model — we will instantiate the training model next.
del _tmp_model, _tokenizer
if device == 'cuda':
    torch.cuda.empty_cache()

## 8. Load the base model and the DataLoader

`paraphrase-multilingual-MiniLM-L12-v2` is ~118 M parameters and fits at
batch 32 with `max_seq_length=128` on a T4. The DataLoader uses
`drop_last=True` so every batch contains the same number of in-batch
negatives (important for MNRL stability).

In [ ]:
model = SentenceTransformer(BASE_MODEL)
model.max_seq_length = MAX_SEQ_LENGTH
model = model.to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {n_params:,}')

In [ ]:
train_dataloader = DataLoader(
    train_examples,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
)
print(f'Batches per epoch: {len(train_dataloader)}')
print(f'Total optimisation steps: {len(train_dataloader) * EPOCHS}')

## 9. Loss function

`MultipleNegativesRankingLoss` is the SBERT default for paired data and
the Pre-Flight recommendation: it draws ~31 negatives per anchor from the
in-batch positives, which is far more contrastive signal than the 3 mined
hard negatives per positive. Robust to label noise — useful given that
Module 2's distant-supervision labels have ~0.65 macro precision on the
eval corpus.

In [ ]:
if MODE == 'pairs_mnrl':
    train_loss = losses.MultipleNegativesRankingLoss(model)
elif MODE == 'triplets':
    train_loss = losses.TripletLoss(model, triplet_margin=0.3)
else:
    raise ValueError(f'Unknown mode {MODE!r}')
print(f'Loss: {type(train_loss).__name__}')

## 10. Information-retrieval evaluator on val

For each validation positive, the evaluator measures whether the encoder
places the gold ESCO concept in the top-k after cosine-similarity ranking
against the entire concept corpus. Tracks MRR@10, Recall@5, Precision@5.
`save_best_model=True` keeps the highest-MRR checkpoint, not the last —
this is the overfitting safety net.

In [ ]:
# Build the IR evaluator: queries = val anchors, corpus = ESCO concept texts,
# relevant = {pair_id: {positive_uri}}.
val_positives = [r for r in val_rows if r['pair_type'] == 'positive']
# Use deterministic query IDs derived from pair_id so re-runs are stable.
queries = {p['pair_id']: build_anchor_text(p) for p in val_positives
           if p['esco_uri'] in concept_text_by_uri}
corpus = dict(concept_text_by_uri)  # uri -> text
relevant_docs = {p['pair_id']: {p['esco_uri']} for p in val_positives
                 if p['esco_uri'] in concept_text_by_uri}

print(f'IR evaluator: {len(queries)} val queries, {len(corpus)} corpus docs.')

ir_evaluator = InformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    name='val',
    mrr_at_k=[10],
    accuracy_at_k=[1, 5, 10],
    precision_recall_at_k=[5, 10],
    map_at_k=[10],
    show_progress_bar=True,
    write_csv=True,
)

## 11. Pre-training (zero-shot) val evaluation

Baseline MRR@10 on val *before* any training — this is the bar that the
final fine-tuned model has to clear by at least 2× (Pre-Flight quantitative
gates). If post-training MRR@10 is < 2× this number, training did not learn
and we re-investigate before downloading the checkpoint.

In [ ]:
baseline_eval = ir_evaluator(model, output_path=OUTPUT_DIR)
print(f'Pre-training val score: {baseline_eval}')

## 12. Train

One call to `model.fit(...)`. The IR evaluator fires every
`EVAL_EVERY_N_STEPS` and `save_best_model=True` keeps the peak-MRR
checkpoint. The block is wrapped in an OOM retry: if the T4 chokes at
`BATCH_SIZE=32`, we halve, empty the CUDA cache, and try once more. If it
still OOMs, we surface the error rather than silently producing a tiny-batch
run that won't match the manifest.

In [ ]:
def _fit(batch_size):
    """Run model.fit at the given batch_size. Returns wall-clock seconds."""
    dl = DataLoader(train_examples, batch_size=batch_size, shuffle=True, drop_last=True)
    steps_per_epoch = len(dl)
    total_steps = steps_per_epoch * EPOCHS
    warmup_steps = int(total_steps * WARMUP_RATIO)
    print(f'Fitting: batch_size={batch_size}, steps/epoch={steps_per_epoch}, '
          f'total_steps={total_steps}, warmup={warmup_steps}')
    t0 = time.time()
    model.fit(
        train_objectives=[(dl, train_loss)],
        evaluator=ir_evaluator,
        epochs=EPOCHS,
        evaluation_steps=EVAL_EVERY_N_STEPS,
        warmup_steps=warmup_steps,
        optimizer_params={'lr': LEARNING_RATE},
        weight_decay=WEIGHT_DECAY,
        output_path=OUTPUT_DIR,
        save_best_model=True,
        show_progress_bar=True,
        checkpoint_save_steps=EVAL_EVERY_N_STEPS,
        checkpoint_save_total_limit=2,
    )
    return time.time() - t0, batch_size

try:
    train_seconds, effective_batch = _fit(BATCH_SIZE)
except torch.cuda.OutOfMemoryError:
    print(f'OOM at batch_size={BATCH_SIZE}; halving and retrying once.')
    torch.cuda.empty_cache()
    train_seconds, effective_batch = _fit(BATCH_SIZE // 2)

print(f'Training done in {train_seconds:.1f}s with effective batch={effective_batch}.')

## 13. Plot training metrics

Reads the CSV that the IR evaluator wrote and plots val-MRR over training
steps. The thesis defence will reference this curve to argue that the
chosen checkpoint is the peak, not the last step.

In [ ]:
import matplotlib.pyplot as plt

# SBERT 3.x writes Trainer-style logs under OUTPUT_DIR/eval/. SBERT 2.x wrote
# Information-Retrieval_evaluation_<name>_results.csv. We try both layouts.
eval_csv_candidates = list((Path(OUTPUT_DIR) / 'eval').glob('*_results.csv')) if (Path(OUTPUT_DIR) / 'eval').exists() else []

if eval_csv_candidates:
    import csv
    eval_csv = eval_csv_candidates[0]
    with eval_csv.open() as fp:
        reader = csv.DictReader(fp)
        rows = list(reader)
    def _g(r, *keys):
        for k in keys:
            if k in r and r[k]:
                return float(r[k])
        return 0.0
    steps = [int(r.get('steps', r.get('step', 0))) for r in rows]
    mrr = [_g(r, 'val_cosine_mrr@10', 'cos_sim-MRR@10', 'cosine-MRR@10') for r in rows]
    r5 = [_g(r, 'val_cosine_recall@5', 'cos_sim-Recall@5', 'cosine-Recall@5') for r in rows]

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(steps, mrr, marker='o', label='val MRR@10')
    ax.plot(steps, r5, marker='s', label='val Recall@5')
    ax.set_xlabel('Training step')
    ax.set_ylabel('Score')
    ax.set_title(f'Fine-tune progress \u2014 {RUN_NAME}')
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(f'{OUTPUT_DIR}/loss_curve.png', dpi=120)
    plt.show()
    print(f'Loss curve saved to {OUTPUT_DIR}/loss_curve.png ({len(rows)} data points)')
else:
    print(f'Note: no eval CSV found under {OUTPUT_DIR}/eval/. The metrics are visible in the Trainer table from cell 28; skipping plot.')

## 14. Post-training evaluation

Loads the best-checkpoint from `OUTPUT_DIR` (which `save_best_model=True`
already wrote) and runs the IR evaluator once more for the manifest record.
Sanity gate: final MRR@10 should be at least 2× the pre-training baseline.

In [ ]:
best_model = SentenceTransformer(OUTPUT_DIR).to(device)
final_eval = ir_evaluator(best_model, output_path=OUTPUT_DIR)
print(f'Post-training val score: {final_eval}')

# SBERT 3.x returns a dict from the IR evaluator (SBERT 2.x returned a float).
# We standardise on the MRR@10 key for the sanity gate; fall back across the
# few naming conventions HF/SBERT has used over versions.
def _mrr_at_10(score):
    if isinstance(score, dict):
        for k in ('val_cosine_mrr@10', 'cos_sim-MRR@10', 'cosine-MRR@10'):
            if k in score:
                return float(score[k])
        return 0.0
    return float(score)

baseline_mrr = _mrr_at_10(baseline_eval)
final_mrr = _mrr_at_10(final_eval)

if baseline_mrr > 0 and final_mrr < 2 * baseline_mrr:
    print(f'WARNING: final MRR@10 ({final_mrr:.4f}) < 2x baseline ({baseline_mrr:.4f}).')
    print('Training did not learn enough. Investigate before downloading the checkpoint.')
else:
    print(f'Sanity gate passed: final MRR@10 ({final_mrr:.4f}) >= 2x baseline ({baseline_mrr:.4f}).')

## 15. Write run manifest and zip the checkpoint

Provenance hash chain: the run manifest carries the bundle SHA from
`bundle_manifest.json` plus every hyperparameter actually used. Together
with the loss-curve PNG this is what the thesis defence references to
answer "how did you get this number?".

In [ ]:
# Helper to JSON-serialise the eval score whether it's a dict (SBERT 3.x) or a float (2.x).
def _serialise_score(score):
    if isinstance(score, dict):
        return {k: float(v) for k, v in score.items()}
    return float(score)

run_manifest = {
    'run_name': RUN_NAME,
    'base_model': BASE_MODEL,
    'mode': MODE,
    'seed': SEED,
    'epochs': EPOCHS,
    'batch_size': effective_batch,
    'requested_batch_size': BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'weight_decay': WEIGHT_DECAY,
    'warmup_ratio': WARMUP_RATIO,
    'max_seq_length': MAX_SEQ_LENGTH,
    'eval_every_n_steps': EVAL_EVERY_N_STEPS,
    'concept_text_format': CONCEPT_TEXT_FORMAT,
    'baseline_val_mrr_at_10': float(baseline_mrr),
    'final_val_mrr_at_10': float(final_mrr),
    'baseline_val_full': _serialise_score(baseline_eval),
    'final_val_full': _serialise_score(final_eval),
    'training_seconds': float(train_seconds),
    'gpu_model': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'torch_version': torch.__version__,
    'sentence_transformers_version': __import__('sentence_transformers').__version__,
    'finished_at_utc': datetime.now(tz=timezone.utc).isoformat(),
    'bundle_manifest': bundle_manifest,
}

manifest_path = f'{OUTPUT_DIR}/{RUN_NAME}_manifest.json'
with open(manifest_path, 'w', encoding='utf-8') as fp:
    json.dump(run_manifest, fp, indent=2, sort_keys=True, ensure_ascii=False)
print(f'Run manifest: {manifest_path}')

In [ ]:
# Zip the OUTPUT_DIR into a single archive Silviu can download.
zip_path = f'{CHECKPOINTS_DIR}/{RUN_NAME}.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    root = Path(OUTPUT_DIR)
    for path in sorted(root.rglob('*')):
        if path.is_file():
            zf.write(path, arcname=path.relative_to(root.parent))
size_mb = os.path.getsize(zip_path) / 1024 / 1024
print(f'Checkpoint zip: {zip_path}  ({size_mb:.1f} MB)')

## 16. Handback 

1. **Download the zip.** Open `MyDrive/hr_helper/checkpoints/` in Drive,
   right-click `{RUN_NAME}.zip`, *Download*. ~500 MB.

2. **Import on the dev box.** From `nlp-service/`:

   ```powershell
   .\.venv\Scripts\Activate.ps1
   python scripts\import_finetuned_model.py --zip <path-to-downloaded.zip> --run-name {RUN_NAME}
   ```

3. **Re-run the baseline.**

   ```powershell
   python scripts\run_finetuned_baseline.py `
     --eval-corpus-dir tests\fixtures\eval_corpus `
     --out-report reports\skill_matcher_finetuned_<YYYYMMDD>.md
   ```

4. **Decide.** Macro F1 ≥ 0.55 → Step 6 (Linker). F1 < 0.55 → open the
   Step 5b Pre-Flight (cross-encoder re-ranker).

**If you need to resume training** (e.g. Colab session dropped mid-fit):
re-run cells 1-7 to rebuild config and data, then in the model-load cell
swap `SentenceTransformer(BASE_MODEL)` for `SentenceTransformer(OUTPUT_DIR)`
and re-fit with reduced EPOCHS. `save_best_model=True` means the best-so-far
is already on Drive.